# 14. Key findings: QoS flows on a shared NR sidelink Mode 2 channel

Five conclusions from the full sweep (`qos_full`: 144 runs, 3 seeds, 15 s each), each with the one
figure that supports it. The figures are also written to `docs/figures/qos/` (not in git) as PNG
and PDF for slides and papers. Notebook 13 has every metric behind them.

**Setup in one line:** NR sidelink Mode 2, 20 MHz at 60 kHz (two 12-RB subchannels), MCS 14,
SUMO mobility; six traffic flows with their PQI targets (platooning, advanced driving, CPM,
emergency trajectories, CAM / VAM, crash mitigation) on highway platooning, urban intersections
and highway CPM scenarios, each at two background densities; resource allocation by sensing only,
+ re-evaluation, + re-evaluation + pre-emption, and for CPMs SB-SPS or dynamic scheduling.

**Superseded packets are left out.** When a newer CPM arrives while the previous one is still
waiting for its resource, the newer one replaces it in the queue and is sent instead
(`KeepOnlyNewestSdu`). The receiver gets the fresher state, so a superseded CPM counts neither as
delivered nor as lost, in every metric of this notebook (about 19 % of the CPMs; only CPMs are
ever superseded). CPM disconnection times need a rerun of the sweep to be recomputed this way and
are not shown until then.

| # | Conclusion | Key number |
|---|---|---|
| 1 | No flow meets its PQI reliability target, even next to the sender and with the best mechanism | PER 2× to 2600× its target within 150 m |
| 2 | On the highway the losses are collisions sensing could have avoided; in the city, buildings | 89–95 % of highway failures are sensable collisions, hidden nodes about 1 % |
| 3 | Background density decides reliability; the mechanism barely moves it | −3 to −16 points from low to high density, a median 0.6 points between mechanisms |
| 4 | Re-evaluation gives a steady gain to SPS flows; pre-emption moves reliability to the SPS flow on top | re-evaluation: CPM +4.0 points; pre-emption: platooning +4.8, crash mitigation −1.3, emergency −1.4 |
| 5 | Variable-size CPMs belong on SB-SPS, not dynamic scheduling | DS loses 15–18 points of CPM delivery within 300 m, 20–22 with re-evaluation |

In [ ]:
import os, sys, json, subprocess
sys.path.insert(0, os.path.abspath("../../tools"))
sys.path.insert(0, os.path.abspath("../../tools/qos"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

mr.apply_chart_style()
os.chdir(mr.ROOT)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

import plots
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

s = plots.Sweep()
print(s.note())
flows, by_distance, causes = s.flows, s.by_distance, s.causes
FIG_DIR = os.path.join(mr.ROOT, "docs", "figures", "qos")
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12.5, "axes.labelsize": 11, "legend.fontsize": 9.5})

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, "%s.%s" % (name, ext)), dpi=200, bbox_inches="tight")

# Same flow colours as notebook 13 (Okabe-Ito, black replaced by wine, checked for colour blindness)
FLOW_STYLE = {
    "crash_mitigation": ("#D55E00", "D"), "emergency_trajectory": ("#882255", "v"),
    "platooning": ("#0072B2", "o"), "advanced_driving": ("#009E73", "s"), "cam": ("#E69F00", "^"),
    "vam": ("#56B4E9", "P"), "cooperative_perception": ("#CC79A7", "X"),
}
FLOW_LABEL = {"crash_mitigation": "Crash mitigation", "emergency_trajectory": "Emergency trajectory",
              "platooning": "Platooning", "advanced_driving": "Advanced driving", "cam": "CAM", "vam": "VAM",
              "cooperative_perception": "CPM"}
FLOW_ORDER = list(FLOW_STYLE)
SCENARIOS = ["highway_platoon", "urban_intersection", "highway_cpm"]
SCENARIO_LABEL = {"highway_platoon": "Highway platooning", "urban_intersection": "Urban intersections",
                  "highway_cpm": "Highway CPM"}
POLICY_LABEL = {"off": "sensing only", "reeval": "+ re-evaluation", "reeval_preempt": "+ re-eval + pre-emption"}
classes = yaml.safe_load(open(os.path.join(mr.ROOT, "scenarios", "qos", "classes.yaml")))["classes"]
T95 = {2: 12.71, 3: 4.30, 4: 3.18, 5: 2.78}

def ci95(v):
    v = np.asarray(v, dtype=float); v = v[~np.isnan(v)]
    if len(v) < 2:
        return (v.mean() if len(v) else np.nan), 0.0
    return v.mean(), T95.get(len(v), 1.96) * v.std(ddof=1) / np.sqrt(len(v))

def logit(p): return np.log(p / (1 - p))
def expit(z): return 1 / (1 + np.exp(-z))

def levels(scenario):
    present = set(flows[flows.scenario == scenario].density)
    return [l for l in ["none", "low", "medium", "high"] if l in present]

def near(max_m):
    """Per run and flow: attempts and in-time decodings within max_m, baseline condition,
    SB-SPS reference variant for the CPM highway."""
    x = by_distance[(by_distance["bin"] < max_m) & (by_distance.condition == "baseline")
                    & by_distance.variant.isin(["default", "sps_max", "dynamic"])].copy()
    x["ok"] = x.qos_pdr * x.attempts
    for c in ("collision", "hidden"):
        x[c] = x[c] * x.attempts
    g = x.groupby(["scenario", "density", "variant", "policy", "seed", "name"])[["attempts", "ok", "collision", "hidden"]].sum().reset_index()
    g["qos_pdr"] = g.ok / g.attempts
    return g

## 1. No flow meets its PQI reliability target, even next to the sender

For every flow, scenario and background level: the packet error rate within **150 m** of the
sender (1 − QoS-PDR, packets decoded within their PDB), with the **best** of the three mechanisms
for that case, divided by the PER target of the flow. 1 means the target is met; every point is to
the right of it. Hollow markers are the lower background level, filled the higher one; the bar is
the 95 % interval over the seeds.

In [ ]:
g = near(150)
g = g[g.variant.isin(["default", "sps_max"])]
rows = []
for (sc, d, name), part in g.groupby(["scenario", "density", "name"]):
    by_policy = part.groupby("policy").qos_pdr.mean()
    best = by_policy.idxmax()
    sel = part[part.policy == best]
    per = (1 - sel.ok / sel.attempts).clip(lower=0.5 / sel.attempts)
    m, h = ci95(np.log10(per / classes[name]["per"]))
    rows.append({"scenario": sc, "density": d, "name": name, "best": best, "ratio": 10 ** m,
                 "lo": 10 ** (m - h), "hi": 10 ** (m + h), "per": per.mean()})
gap = pd.DataFrame(rows)

order = [(sc, n) for sc in SCENARIOS for n in FLOW_ORDER if ((gap.scenario == sc) & (gap.name == n)).any()]
fig, ax = plt.subplots(figsize=(10, 6.2))
for i, (sc, name) in enumerate(order):
    y = len(order) - 1 - i
    col, mk = FLOW_STYLE[name]
    part = gap[(gap.scenario == sc) & (gap.name == name)]
    xs = []
    for _, r in part.iterrows():
        high = r.density == "high"
        ax.errorbar(r.ratio, y, xerr=[[r.ratio - r.lo], [r.hi - r.ratio]], fmt=mk, color=col, ms=8,
                    mfc=col if high else "white", mew=1.6, elinewidth=1, capsize=2, zorder=3)
        xs.append(r.ratio)
    if len(xs) == 2:
        ax.plot(xs, [y, y], color=col, lw=1, alpha=0.5, zorder=2)
ax.axvline(1, color="#1f1e1c", lw=1.2, ls="--")
ax.text(1.08, len(order) - 0.6, "target met\n← here", fontsize=9.5, va="top", color="#1f1e1c")
ax.set_xscale("log"); ax.set_xlim(0.7, 1e4)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(["%s · %s" % (FLOW_LABEL[n], SCENARIO_LABEL[sc]) for sc, n in order][::-1], fontsize=9.5)
ax.set_xlabel("measured PER within 150 m ÷ PER target (best mechanism, log scale)")
ax.legend(handles=[Line2D([], [], ls="", marker="o", mfc="white", mec="#52514e", ms=8, label="low / medium background"),
                   Line2D([], [], ls="", marker="o", color="#52514e", ms=8, label="high background")],
          loc="lower right", frameon=True)
ax.set_title("No flow meets its PQI reliability target, even within 150 m", loc="left")
plt.tight_layout(); save(fig, "1_target_gap"); plt.show()
print("closest to its target: %s, %s, %s background: %.1f× (PER %.4f)" % tuple(
    gap.sort_values("ratio").iloc[0][["name", "scenario", "density", "ratio", "per"]]))
print("furthest: %s, %s, %s background: %.0f×" % tuple(gap.sort_values("ratio").iloc[-1][["name", "scenario", "density", "ratio"]]))

**Read it as:** the strict flows (PER 10⁻⁴, 10⁻⁵) miss by two to three orders of magnitude even at
low density and short range: platooning and crash mitigation reach a PER of about 1.4 × 10⁻³
within 150 m at low background. Blind retransmissions (1 for 10⁻⁴, 2 for 10⁻⁵) do not close that gap.
The closest cases, advanced driving and CPMs at low background, are still about twice their
10⁻² target; CPMs elsewhere miss it by 6 to 10 times.

## 2. Highway losses are collisions sensing could have avoided; urban losses are buildings

How many receptions fail within 300 m of the sender, and why. Every failed reception has one
cause; all flows and the three mechanisms pooled, baseline condition. **The length of a bar is the
share of all receptions that fail**, and each segment is the share lost to one cause, so the
segments add up to the bar. The highways and the urban grid have different scales (top and
bottom); the table under the figure gives every number.

In [ ]:
GROUPS = {
    "sensing-visible collision": "collision, interferer sensable",
    "hidden-node collision": "collision, hidden interferer",
    "propagation": "signal too weak", "below sensitivity": "signal too weak",
    "oversize drop": "never sent", "intra-UE conflict": "never sent", "never transmitted": "never sent",
    "half duplex": "half duplex", "late (decoded after the PDB)": "late", "injected burst": "injected burst",
    "decoded in time": "decoded",
}
GROUP_ORDER = ["collision, interferer sensable", "collision, hidden interferer", "signal too weak", "never sent", "half duplex"]
GROUP_COLOR = {"collision, interferer sensable": "#B2182B", "collision, hidden interferer": "#EF8A62",
               "signal too weak": "#2166AC", "never sent": "#B8B7B0", "half duplex": "#6B6A66"}
c = causes[(causes.condition == "baseline") & (causes["bin"] < 300) & causes.variant.isin(["default", "sps_max"])].copy()
c["group"] = c.cause.map(GROUPS)
counts = c.groupby(["scenario", "density", "group"])["count"].sum().unstack(fill_value=0)
rate = 100 * counts.div(counts.sum(axis=1), axis=0)          # % of all receptions within 300 m
GROUP_ORDER = [g for g in GROUP_ORDER if g in rate.columns and rate[g].max() > 0]   # causes that occur
rate = rate[GROUP_ORDER]
rate["failed"] = rate.sum(axis=1)

panels = [["highway_platoon", "highway_cpm"], ["urban_intersection"]]
heights = [sum(len(levels(sc)) for sc in p) for p in panels]
fig, axes = plt.subplots(2, 1, figsize=(10.5, 5.6), gridspec_kw={"height_ratios": heights})
for ax, scen in zip(axes, panels):
    rows = [(sc, d) for sc in scen for d in levels(sc)]
    part = rate.reindex(rows)
    xmax = part.failed.max() * 1.18
    left = np.zeros(len(rows))
    for grp in GROUP_ORDER:
        vals = part[grp].values
        ax.barh(range(len(rows)), vals, left=left, color=GROUP_COLOR[grp], edgecolor="white", lw=1.2, height=0.7,
                label=grp if ax is axes[0] else None)
        for i, (v, l) in enumerate(zip(vals, left)):
            if v / xmax >= 0.05:
                ax.text(l + v / 2, i, "%.1f" % v, ha="center", va="center", fontsize=9.5,
                        color="#1f1e1c" if grp in ("never sent", "collision, hidden interferer") else "white")
        left += vals
    for i, total in enumerate(part.failed.values):
        ax.text(total + xmax * 0.01, i, "%.1f %% fail" % total, va="center", fontsize=10, color="#1f1e1c")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(["%s, %s background" % (SCENARIO_LABEL[sc], d) for sc, d in rows])
    ax.invert_yaxis(); ax.set_xlim(0, xmax); ax.grid(axis="y", visible=False)
    ax.set_xlabel("receptions within 300 m that fail [%]   (segments: % lost to each cause)", fontsize=10)
fig.suptitle("How many receptions fail within 300 m, and why", x=0.01, ha="left", fontsize=12.5)
axes[0].set_title("Highways (axis 0 to %.0f %%)" % axes[0].get_xlim()[1], loc="left", fontsize=10.5)
axes[1].set_title("Urban grid (axis 0 to %.0f %%, a different scale)" % axes[1].get_xlim()[1], loc="left", fontsize=10.5)
fig.legend(*axes[0].get_legend_handles_labels(), ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.06), frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); save(fig, "2_loss_causes"); plt.show()
rate.reindex([(sc, d) for sc in SCENARIOS for d in levels(sc)]).round(2).rename_axis(columns="% of receptions within 300 m")

**Why each cause happens:**

* **Collision, interferer sensable** (platooning highway: 1.9 % of receptions at low background,
  11.7 % at high; CPM highway 8.1 % → 12.2 %). The sender could hear the interferer, yet both used
  the same resource. Sensing only knows reservations announced in the last 1.1 s, so it misses
  the first transmission of every dynamically scheduled packet and every new SPS selection, and two
  UEs selecting at the same moment both see the same resource free. On a loaded channel the
  selection itself reuses occupied resources: when fewer than 20 % of the candidates survive the
  RSRP exclusion, TS 38.214 §8.1.4 raises the threshold by 3 dB and tries again, until the UE
  accepts resources reserved by UEs it hears weakly, and a receiver near those UEs then sees a
  collision. With two subchannels per slot every overlap covers the whole slot. All three grow with
  the number of UEs, hence the sixfold increase from low to high background.
* **Collision, hidden interferer** (at most 0.18 % on the highways, 0.5 to 1.8 % in the urban grid).
  It needs an interferer the receiver hears but the sender does not. At 23 dBm on a straight
  line-of-sight highway the sender hears almost everyone its receivers hear, so it is rare. In the
  urban grid buildings hide UEs around a corner from the sender, so it is ten times more frequent,
  but still small next to the signal loss.
* **Signal too weak** (0.1 to 0.3 % on the highways, 45 to 48 % in the urban grid). The received
  power is under the −103.5 dBm sensitivity (about 9 % of the urban receptions) or the SINR fails
  with no overlapping transmission (36 to 39 %). Within 300 m in the grid most receivers are around
  a corner, where the TR 37.885 NLOS pathloss grows with 30 log10 d instead of about 17 log10 d in
  the street canyon, plus 4 dB shadowing. On the highway the receivers within 300 m are in line of
  sight.
* **Half duplex** (0.1 to 0.2 %). A UE cannot receive in a slot in which it transmits; each UE
  transmits in only a small share of the slots.

So on the highways the losses that grow with density are the sensable collisions, 89 to 95 % of
the failures: a resource choice problem, and hidden nodes play almost no part even on a collinear
road. In the urban grid the weak signal is 95 to 98 % of the failures: the buildings decide, and no
resource allocation can change that.

**Superseded CPMs, left out of every metric.** A CPM waits for its resource up to its 100 ms PDB
(SB-SPS: the next reserved slot of the 100 ms cycle; dynamic scheduling: the slot it picked in its
100 ms selection window), while the next CPM comes 50 ms + Exp(50 ms) later. When the next one
comes first, it replaces the waiting one in the queue and is sent in its place. For a uniform
wait in [0, 100) ms this happens with probability 0.5 e⁻¹ ≈ 18.4 %; the sweep gives about 19 %
under SB-SPS at every density (table below), so it is a property of the CPM generation and the
PDB, not of the load. The receiver gets the newer, fresher CPM, so these are not counted as
failures here.

In [ ]:
cpm = by_distance[(by_distance.condition == "baseline") & (by_distance["bin"] < 300)
                  & by_distance.variant.isin(["default", "sps_max", "dynamic"]) & (by_distance.name == "cooperative_perception")]
cpm = cpm.groupby(["scenario", "density", "variant"])[["attempts", "superseded"]].sum()
(100 * cpm.superseded / (cpm.attempts + cpm.superseded)).round(1).rename(
    "superseded, % of CPM receptions within 300 m (not counted in any metric)").to_frame()

## 3. Background density decides reliability; the mechanism barely moves it

QoS-PDR within 300 m for every flow, at both background levels, for each of the three mechanisms
(marker shape). Hollow = lower background, filled = higher background. The three mechanisms of one
level sit almost on top of each other; the jump between levels is several times larger.

In [ ]:
g = near(300)
g = g[g.variant.isin(["default", "sps_max"])]
mean = g.groupby(["scenario", "density", "name", "policy"]).qos_pdr.mean()
POLICY_MARKER = {"off": "o", "reeval": "s", "reeval_preempt": "^"}
order = [(sc, n) for sc in SCENARIOS for n in FLOW_ORDER if n in set(g[g.scenario == sc].name)]
fig, ax = plt.subplots(figsize=(10, 6.2))
spreads, gaps = [], []
for i, (sc, name) in enumerate(order):
    y = len(order) - 1 - i
    col, _ = FLOW_STYLE[name]
    centres = {}
    for d in levels(sc):
        if (sc, d, name) not in mean.index.droplevel("policy"):
            continue
        vals = mean.loc[(sc, d, name)]
        high = d == "high"
        for p, v in vals.items():
            ax.scatter(100 * v, y, marker=POLICY_MARKER[p], s=55, facecolor=col if high else "white", edgecolor=col, lw=1.5, zorder=3)
        centres[d] = 100 * vals.mean()
        spreads.append(100 * (vals.max() - vals.min()))
    if len(centres) == 2:
        a, b = list(centres.values())
        ax.plot([a, b], [y, y], color=col, lw=1, alpha=0.45, zorder=2)
        gaps.append(abs(a - b))
ax.set_yticks(range(len(order)))
ax.set_yticklabels(["%s · %s" % (FLOW_LABEL[n], SCENARIO_LABEL[sc]) for sc, n in order][::-1], fontsize=9.5)
ax.set_xlabel("QoS-PDR within 300 m [%]")
ax.legend(handles=[Line2D([], [], ls="", marker=POLICY_MARKER[p], mfc="white", mec="#52514e", ms=8, label=POLICY_LABEL[p]) for p in POLICY_MARKER]
          + [Line2D([], [], ls="", marker="o", mfc="white", mec="#52514e", ms=8, label="low / medium background"),
             Line2D([], [], ls="", marker="o", color="#52514e", ms=8, label="high background")], loc="upper left", frameon=True)
ax.set_title("Density moves reliability far more than the mechanism does", loc="left")
plt.tight_layout(); save(fig, "3_density_vs_mechanism"); plt.show()
print("low -> high background: %.1f to %.1f points; between mechanisms at one level: median %.1f, max %.1f points"
      % (min(gaps), max(gaps), np.median(spreads), max(spreads)))

**Read it as:** going from the lower to the higher background costs every flow 3 to 16 points of
delivery within 300 m. Switching mechanism moves a flow by a median of well under one point; the
exceptions are platooning with pre-emption on the dense highway (6.3 points) and CPMs with
re-evaluation (about 4 points), both in section 4. On a channel this
loaded, no resource selection rule substitutes for capacity.

## 4. Re-evaluation gives SPS flows a steady gain; pre-emption shifts reliability to the SPS flow on top

Dense platooning highway (20 veh/km/lane), paired differences on the same seed and trace, QoS-PDR
within 300 m. Solid: re-evaluation against sensing only. Hatched: pre-emption added on top of
re-evaluation. Flows in priority order (1 = highest); the bar is the 95 % interval over the seeds.

In [ ]:
g = near(300)
g = g[(g.scenario == "highway_platoon") & (g.density == "high")]
piv = g.pivot_table(index=["seed", "name"], columns="policy", values="qos_pdr")
d_re = 100 * (piv["reeval"] - piv["off"])
d_pre = 100 * (piv["reeval_preempt"] - piv["reeval"])
names = sorted(set(g.name), key=lambda n: classes[n]["priority"])
fig, ax = plt.subplots(figsize=(10, 4.8))
summary = []
for i, name in enumerate(names):
    col, _ = FLOW_STYLE[name]
    for k, (d, lab) in enumerate([(d_re, "re-evaluation"), (d_pre, "pre-emption")]):
        m, h = ci95(d.xs(name, level="name"))
        ax.bar(i + 0.38 * (k - 0.5), m, 0.36, yerr=h, capsize=3, color=col if k == 0 else "white",
               edgecolor=col, hatch="" if k == 0 else "////", lw=1.4, error_kw=dict(ecolor="#1f1e1c", lw=1))
        summary.append({"flow": FLOW_LABEL[name], "priority": classes[name]["priority"], "contribution": lab,
                        "delta [points]": round(m, 2), "95 % half-width": round(h, 2)})
ax.axhline(0, color="#1f1e1c", lw=0.9)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(["%s\npriority %d · %s" % (FLOW_LABEL[n], classes[n]["priority"],
                    "SPS" if classes[n]["scheduling"] == "sps" else "DS") for n in names], fontsize=9.5)
ax.set_ylabel("Δ QoS-PDR within 300 m [points]")
ax.legend(handles=[Patch(facecolor="#52514e", label="re-evaluation (vs sensing only)"),
                   Patch(facecolor="white", edgecolor="#52514e", hatch="////", label="pre-emption (added to re-evaluation)")],
          loc="lower left", frameon=True)
ax.set_title("Pre-emption moves reliability to platooning, the highest-priority SPS flow", loc="left")
plt.tight_layout(); save(fig, "4_reeval_preemption"); plt.show()

dt = flows[(flows.scenario == "highway_platoon") & (flows.name == "platooning") & (flows.condition == "baseline")]
dt = dt.pivot_table(index=["density", "seed"], columns="policy", values="dt_p99_ms")
for d in ("low", "high"):
    m, h = ci95((dt.loc[d]["reeval_preempt"] - dt.loc[d]["off"]).values)
    print("platooning DT p99, %s background: %+.0f ± %.0f ms with re-evaluation + pre-emption (from %.0f ms)"
          % (d, m, h, dt.loc[d]["off"].mean()))
pd.DataFrame(summary).pivot_table(index=["priority", "flow"], columns="contribution", values="delta [points]")

**Read it as:** re-evaluation adds 0.7 to 1.5 points for platooning, advanced driving and CAMs,
and 4.0 points for the CPMs, whose aperiodic, variable-size selections it catches most often. It
costs crash mitigation 1.2 points. Pre-emption gives platooning +4.8 points. The flows it pre-empts
lose 0.6 to 1.4 points each, within the noise of three seeds, and, unexpectedly, crash mitigation
and emergency trajectories lose 1.3 and 1.4 points although they rank *above* platooning. Those two use dynamic scheduling, so they hold no announced
reservation that pre-emption could protect, and the extra reselections it triggers hit them.
Pre-emption is a priority transfer between flows, not a capacity gain: the mean over all flows
does not move (−0.1 ± 0.3 points).

Averages also hide the tail: at low background, re-evaluation + pre-emption lengthen the 99th
percentile disconnection time of platooning by about 250 ms (about +50 %), while its mean
delivery does not change.

## 5. Variable-size CPMs belong on SB-SPS, not on dynamic scheduling

Highway CPM scenario, every vehicle sending CAMs and variable-size CPMs (200, 400 or 600 B, 50 ms +
Exp(50 ms)). CPM QoS-PDR against distance with the CPMs on SB-SPS (reserving for the largest size)
or on dynamic scheduling (a new selection for every CPM), sensing only, both densities. The band is
the 95 % interval over the seeds.

In [ ]:
bd = by_distance[(by_distance.scenario == "highway_cpm") & (by_distance.condition == "baseline")
                 & (by_distance.policy == "off") & (by_distance.name == "cooperative_perception")].copy()
bd["x"] = bd["bin"] + 25
col = FLOW_STYLE["cooperative_perception"][0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, d in zip(axes, levels("highway_cpm")):
    for variant, ls, lab in (("sps_max", "-", "SB-SPS"), ("dynamic", "--", "dynamic scheduling")):
        part = bd[(bd.density == d) & (bd.variant == variant)]
        floor = 0.5 / part.attempts
        part = part.assign(lv=logit(part.qos_pdr.clip(lower=floor, upper=1 - floor)))
        stats = part.groupby("x").agg(m=("qos_pdr", "mean"), lm=("lv", "mean"), ls_=("lv", "std"), n=("lv", "count"))
        h = stats.n.map(T95).fillna(1.96) * stats.ls_ / np.sqrt(stats.n)
        ax.plot(stats.index, 100 * stats.m, color=col, ls=ls, lw=2.2, label=lab)
        ax.fill_between(stats.index, 100 * expit(stats.lm - h), 100 * expit(stats.lm + h), color=col, alpha=0.18, lw=0)
    ax.set_title("%s background (%d veh/km/lane)" % (d, {"medium": 20, "high": 30}[d]), fontsize=11)
    ax.set_xlabel("distance from the transmitter [m]")
axes[0].set_ylabel("CPM QoS-PDR [%]"); axes[0].legend(frameon=True)
fig.suptitle("CPMs on SB-SPS against dynamic scheduling", x=0.01, ha="left", fontsize=12.5)
plt.tight_layout(); save(fig, "5_cpm_sps_vs_ds"); plt.show()

g = near(300)
g = g[(g.scenario == "highway_cpm") & (g.name == "cooperative_perception")]
piv = g.pivot_table(index=["density", "policy", "seed"], columns="variant", values="qos_pdr")
delta = 100 * (piv["dynamic"] - piv["sps_max"])
out = delta.groupby(["density", "policy"]).apply(lambda v: "%+.1f ± %.1f" % ci95(v.values)).unstack()
out.rename(columns=POLICY_LABEL).rename_axis(columns="DS − SB-SPS, CPM QoS-PDR within 300 m [points]")

**Read it as:** a CPM on dynamic scheduling is selected blind for every packet and announces
nothing the other UEs could avoid, so it collides more: 15 to 18 points less delivery within
300 m than on SB-SPS, at every distance and both densities. Re-evaluation does not rescue it: it
gives SB-SPS CPMs 3 to 4 points and DS CPMs nothing (−0.7 to −1.0), so the gap grows to 20 to 22
points. Reserving for the largest CPM wastes some of the reservation on small CPMs but is still the
better choice here.

## What these results do not show

* **Three seeds.** Effects of a point or less are inside the noise; the conclusions above are the
  ones the three seeds agree on.
* **Blind retransmissions only**, no HARQ feedback, and the SCI announces only the next copy.
* **SL priorities are an assumption** (the PQI table gives none: crash mitigation 1 … CPM 6), and
  the RSRP threshold does not depend on the priority pair (`PriorityAware` off).
* **Emergency trajectories** exist only at high background, from 1 to 5 vehicles per seed.
* **Two subchannels at 20 MHz**: every packet above about 300 B takes the whole slot, and reserving
  for a median CPM size cannot be told apart from reserving for the largest.
* **Superseded CPMs are left out** (about 19 % of the CPMs, see section 2). Counted as losses
  instead, CPM delivery would be 10 to 17 points lower and the conclusions would not change. CPM
  disconnection times are not shown until the sweep is rerun with the current `metrics.py`.
* No run without background traffic; no closed-loop platoon control.